<a href="https://colab.research.google.com/github/Vishrutha333/BIS_LAB/blob/main/GeneExpressionAlgorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import random

def fitness(x):
    return (x * x) + (x * x) - (x + x)


class GEP:

    def __init__(self, grammar, bounds, len_head, iterations,
                 pop_size, prob_crossover, num_trials):

        self.grammar = grammar
        self.bounds = bounds
        self.len_head = len_head
        self.len_tail = len_head + 1
        self.iterations = iterations
        self.pop_size = pop_size
        self.prob_crossover = prob_crossover
        self.num_trials = num_trials

        self.population = [
            Individual(self.random_genome(), self)
            for _ in range(self.pop_size)
        ]

        self.best = min(
            self.population,
            key=lambda x: x.fitness
        )

    def random_genome(self):
        genome = []

        for i in range(self.len_head):
            selection = random.choice(
                self.grammar['FUNC'] + self.grammar['TERM']
            )
            genome.append(selection)

        for i in range(self.len_tail):
            genome.append(
                random.choice(self.grammar['TERM'])
            )

        return ''.join(genome)

    def mapping(self, genome):
        index = 0

        def build():
            nonlocal index

            gene = genome[index]
            index += 1

            if gene in self.grammar['FUNC']:
                left = build()
                right = build()
                return [gene, left, right]

            return gene

        return build()

    def tree_to_string(self, tree):
        if isinstance(tree, str):
            return tree

        left = self.tree_to_string(tree[1])
        right = self.tree_to_string(tree[2])

        operator = tree[0]

        if operator == 'P':
            return "({} ** {})".format(left, right)

        return "({} {} {})".format(left, operator, right)

    def cost(self, program):
        errors = 0.0

        for i in range(self.num_trials):
            x = random.uniform(
                self.bounds[0],
                self.bounds[1]
            )

            try:
                score = eval(
                    program,
                    {"__builtins__": {}},
                    {"x": x}
                )

                errors += abs(score - fitness(x))

            except (ZeroDivisionError, OverflowError,
                    ValueError, TypeError):
                return float('inf')

            if not np.isfinite(errors):
                return float('inf')

        return errors / self.num_trials

    def point_mutation(self, genome):
        child = list(genome)
        rate = 1.0 / len(genome)

        for i in range(len(child)):
            if random.random() < rate:

                if i < self.len_head:
                    child[i] = random.choice(
                        self.grammar['FUNC'] +
                        self.grammar['TERM']
                    )
                else:
                    child[i] = random.choice(
                        self.grammar['TERM']
                    )

        return ''.join(child)

    def crossover(self, parent1, parent2):

        if random.random() < self.prob_crossover:
            point = random.randint(1, len(parent1) - 1)

            return (
                parent1[:point] + parent2[point:]
            )

        return parent1

    def reproduce(self, selected):
        children = []

        for i in range(0, len(selected) - 1, 2):
            a = selected[i].genome
            b = selected[i + 1].genome

            child = self.crossover(a, b)
            child = self.point_mutation(child)

            children.append(Individual(child, self))

        return children

    def binary_tournament(self):
        a, b = random.sample(self.population, 2)

        return a if a.fitness < b.fitness else b

    def run(self):

        for gen in range(1, self.iterations + 1):

            selected = [
                self.binary_tournament()
                for _ in range(self.pop_size)
            ]

            children = self.reproduce(selected)

            children.sort(key=lambda x: x.fitness)

            if children[0].fitness < self.best.fitness:
                self.best = children[0]

            self.population = sorted(
                children + [self.best],
                key=lambda x: x.fitness
            )[:self.pop_size]

            if gen % 50 == 0 or gen == 1:
                print("Generation:", gen)
                print("Best Program:", self.best.program)
                print("Fitness:", self.best.fitness)
                print()

            if self.best.fitness < 0.000001:
                break

        return self.best


class Individual:

    def __init__(self, genome, gep):
        self.genome = genome
        self.expression = gep.mapping(genome)
        self.program = gep.tree_to_string(self.expression)
        self.fitness = gep.cost(self.program)


if __name__ == '__main__':

    random.seed(42)
    np.random.seed(42)

    GRAMMAR = {
        'FUNC': ['+', '-', '*', '/', 'P'],
        'TERM': ['x']
    }

    BOUNDS = [-1.0, 10.0]
    NUM_TRIALS = 30
    LEN_HEAD = 30
    ITERATIONS = 200
    POP_SIZE = 100
    PROB_CROSS = 0.85

    gep = GEP(
        GRAMMAR,
        BOUNDS,
        LEN_HEAD,
        ITERATIONS,
        POP_SIZE,
        PROB_CROSS,
        NUM_TRIALS
    )

    best = gep.run()

    print("FINAL RESULT")
    print("Best Expression:", best.program)
    print("Best Fitness:", best.fitness)


Generation: 1
Best Program: (((((((((x - (x - ((((x + ((((x * x) + (((x / (x * x)) + x) - x)) * x) * x)) * x) - x) * x))) / x) + x) / x) / x) - x) + x) + x) / x)
Fitness: 20.658267640269305

FINAL RESULT
Best Expression: ((x + ((((((x - (x - ((((((((((x + (x - (x / ((x / x) * x)))) * x) - x) * x) * x) - x) + x) * x) - x) * x))) / x) + x) / x) / x) - x)) / x)
Best Fitness: 5.020983628867271e-15
